<h1>🏗️ Final Year Project (PFE) – Database Verification (Part 3)</h1>
<h2>Financial Analysis and Forecasting for an Engineering Consulting Firm</h2>

<hr>

<h3>⚙️ Project Overview</h3>
<p>
This notebook performs <strong>verification and validation</strong> of the staging area and data warehouse databases created for the financial analysis project. 
It ensures that the <strong>staging_area</strong> database and the subsequent <strong>data warehouse</strong> schema are properly defined, accessible, and ready for ETL and reporting processes.
</p>

<hr>

<h3>🎯 Objectives</h3>
<ul>
  <li>Verify connection to <strong>staging_area</strong> (SA) Postgres database</li>
  <li>Verify existence and schema of <strong>staging_data</strong> table</li>
  <li>Verify connection to the <strong>data warehouse</strong> (DW) database</li>
  <li>Check that dimension and fact tables are present and structured correctly</li>
  <li>Perform basic row counts and data consistency checks</li>
</ul>

<hr>


<h3>📌 Data Sources</h3>
<ul>
  <li><strong>Staging Database:</strong> <code>sales_sa</code></li>
  <li><strong>Data Warehouse:</strong> <code>sales_dw</code></li>
</ul>

<hr>

<h3>📁 Notebook Summary</h3>
<p>
This notebook ensures that both the staging and warehouse databases are properly configured and ready for ETL and reporting. The key steps include:
<ul>
  <li><strong>1. Define Database Connection:</strong> Establish connectivity with SA and DW databases.</li>
  <li><strong>2. Verify SA Database:</strong> Confirm existence of <code>sales_sa</code> and perform sanity checks.</li>
  <li><strong>3. Verify DW Database:</strong> Confirm existence of dimension and fact tables, verify schema and data integrity.</li>
</ul>

</p>

<hr>

<strong>👨‍💻 Author:</strong> 
<a href="https://linkedin.com/in/asma-bouach">Asma Bouach</a><br>
<strong>📅 Project Period:</strong> Feb-May 2025<br>
<strong>🎓 Thesis Defense:</strong> July 2025<br>
<strong>🔄 Last Updated:</strong> December 2025 (Refactored with synthetic sata)
</p>

### 1. Define Database Connection

In [1]:
import warnings
import os
import yaml
import psycopg2
import pandas as pd
from dotenv import load_dotenv

# Ignore the specific UserWarning from pandas about DBAPI2 connections
warnings.filterwarnings("ignore", category=UserWarning, message="pandas only supports SQLAlchemy")

In [2]:
# Load environment variables from .env file
load_dotenv()

# Load database configuration from YAML file
with open("../configs/db.yml", "r") as f:
    db_config = yaml.safe_load(f)

# Function to connect to the database
def connect_db(env="staging"):
    return psycopg2.connect(
        host=db_config[env]["host"],           # Database host address
        database=db_config[env]["database"],   # Database name
        user=db_config[env]["user"],           # Database username
        password=os.getenv(db_config[env]["password_env"]),  # Password from environment variable
        port=db_config[env]["port"]            # Database port
    )

In [3]:
# Def to Query a DB and Return a DataFrame
def query_df(sql, env="staging"):
    conn = connect_db(env)
    df = pd.read_sql(sql, conn)
    conn.close()
    return df

### 2. Verify SA database

In [4]:
# Query for Table names in the SA database
df = query_df("""SELECT table_name
FROM information_schema.tables
WHERE table_schema='public'
ORDER BY table_name;
""", "staging")
df

,table_name
0,client
1,data_sales
2,date
3,invoice
4,sales


In [5]:
# Query for the First 5 rows of sales
df = query_df("SELECT * FROM sales LIMIT 5;", "staging")
df

,sales_id,client_id,invoice_number,date_id,gross_sales,discount,net_sales,cog,profit,vat,stamp_duty,total_amount,client_category
0,1,C812,OR2300000003,20230101,2015.65,0.0,2015.65,1670.28,345.37,382.97,1.0,2399.62,Private Organization
1,2,C2773,OR2300000004,20230101,1531.69,0.0,1531.69,976.30,555.39,291.02,1.0,1823.71,Private Organization
2,3,C2264,OR2300000005,20230101,1746.59,0.0,1746.59,1084.43,662.16,331.85,1.0,2079.44,Private Organization
3,4,C2404,OR2300000001,20230101,19066.38,0.0,19066.38,8251.99,10814.39,3622.61,1.0,22689.99,Private Organization
4,5,C2592,OR2300000002,20230101,14487.86,0.0,14487.86,8670.07,5817.79,2752.69,1.0,17241.55,Private Organization


In [6]:
# Query to Show column names
df = query_df("SELECT * FROM sales LIMIT 1;", "staging")
list(df.columns)

['sales_id',
 'client_id',
 'invoice_number',
 'date_id',
 'gross_sales',
 'discount',
 'net_sales',
 'cog',
 'profit',
 'vat',
 'stamp_duty',
 'total_amount',
 'client_category']

In [7]:
# Query to Distinct clients
query_df("SELECT COUNT(DISTINCT client_id) AS unique_clients FROM sales;", "staging")

,unique_clients
0,2804


In [8]:
# Example: query staging database
conn = connect_db("staging")
cursor = conn.cursor()
cursor.execute("SELECT DISTINCT client_id FROM client LIMIT 10;")
print(cursor.fetchall())
cursor.close()
conn.close()

[('C0',), ('C1',), ('C10',), ('C100',), ('C1000',), ('C1001',), ('C1002',), ('C1003',), ('C1004',), ('C1005',)]


### 3. Verify DW database

In [9]:
# Query for Table names in DW the database
query_df("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema='public'
    ORDER BY table_name;
""", "dw")

,table_name
0,dim_client
1,dim_date
2,dim_invoice
3,fact_sales


In [10]:
query_df("SELECT COUNT(*) AS total_sales FROM fact_sales;", "dw")

,total_sales
0,26507


In [11]:
query_df("SELECT * FROM fact_sales LIMIT 10;", "dw")

,sales_id,client_id,invoice_number,date_id,gross_sales,discount,net_sales,cog,profit,vat,stamp_duty,total_amount,client_category
0,1,C812,OR2300000003,20230101,2015.65,0.00,2015.65,1670.28,345.37,382.97,1.0,2399.62,Private Organization
1,2,C2773,OR2300000004,20230101,1531.69,0.00,1531.69,976.30,555.39,291.02,1.0,1823.71,Private Organization
2,3,C2264,OR2300000005,20230101,1746.59,0.00,1746.59,1084.43,662.16,331.85,1.0,2079.44,Private Organization
3,4,C2404,OR2300000001,20230101,19066.38,0.00,19066.38,8251.99,10814.39,3622.61,1.0,22689.99,Private Organization
4,5,C2592,OR2300000002,20230101,14487.86,0.00,14487.86,8670.07,5817.79,2752.69,1.0,17241.55,Private Organization
5,6,C1565,OR2300000006,20230101,1453.98,16.95,1437.03,836.33,600.70,273.04,1.0,1711.07,Private Organization
6,7,C1839,RE2300000001,20230101,-433.25,0.00,-433.25,0.00,-433.25,-82.32,0.0,-515.57,State Organization
7,8,C2357,OR2300000013,20230102,2106.97,0.00,2106.97,714.75,1392.22,400.33,1.0,2508.30,Private Organization
8,9,C2263,IC2300000002,20230102,2355.93,0.00,2355.93,1024.81,1331.12,447.63,1.0,2804.56,Insurance Agency
9,10,C2422,OR2300000011,20230102,566.68,0.00,566.68,309.39,257.29,107.67,1.0,675.35,Private Organization


In [12]:
query_df("""
    SELECT client_category, COUNT(*) AS Row_Count
    FROM fact_sales
    GROUP BY client_category
    ORDER BY client_category;
""", "dw")

,client_category,row_count
0,Insurance Agency,3986
1,Private Organization,17948
2,State Organization,3253
3,Walk-In Client,1320


In [13]:
query_df("""SELECT *
FROM fact_sales f
JOIN dim_date d ON f.date_id = d.date_id
LIMIT 10;
""", "dw")

,sales_id,client_id,invoice_number,date_id,gross_sales,discount,net_sales,cog,profit,vat,stamp_duty,total_amount,client_category,date_id,date
0,1,C812,OR2300000003,20230101,2015.65,0.00,2015.65,1670.28,345.37,382.97,1.0,2399.62,Private Organization,20230101,2023-01-01
1,2,C2773,OR2300000004,20230101,1531.69,0.00,1531.69,976.30,555.39,291.02,1.0,1823.71,Private Organization,20230101,2023-01-01
2,3,C2264,OR2300000005,20230101,1746.59,0.00,1746.59,1084.43,662.16,331.85,1.0,2079.44,Private Organization,20230101,2023-01-01
3,4,C2404,OR2300000001,20230101,19066.38,0.00,19066.38,8251.99,10814.39,3622.61,1.0,22689.99,Private Organization,20230101,2023-01-01
4,5,C2592,OR2300000002,20230101,14487.86,0.00,14487.86,8670.07,5817.79,2752.69,1.0,17241.55,Private Organization,20230101,2023-01-01
5,6,C1565,OR2300000006,20230101,1453.98,16.95,1437.03,836.33,600.70,273.04,1.0,1711.07,Private Organization,20230101,2023-01-01
6,7,C1839,RE2300000001,20230101,-433.25,0.00,-433.25,0.00,-433.25,-82.32,0.0,-515.57,State Organization,20230101,2023-01-01
7,8,C2357,OR2300000013,20230102,2106.97,0.00,2106.97,714.75,1392.22,400.33,1.0,2508.30,Private Organization,20230102,2023-01-02
8,9,C2263,IC2300000002,20230102,2355.93,0.00,2355.93,1024.81,1331.12,447.63,1.0,2804.56,Insurance Agency,20230102,2023-01-02
9,10,C2422,OR2300000011,20230102,566.68,0.00,566.68,309.39,257.29,107.67,1.0,675.35,Private Organization,20230102,2023-01-02


<h3>📦 Data Warehouse Star Schema Overview</h3>

<p>
The data warehouse adopts a <strong>Star Schema</strong> to normalize and organize the data efficiently. 
The central <strong>fact table</strong> stores measurable business metrics (e.g., sales, profit, discounts), 
while surrounding <strong>dimension tables</strong> capture descriptive attributes related to invoices, clients and time.
</p>


<h4>Fact Table: <code>fact_sales</code> Rows: ~26,507 (one per invoice)</h4>
<table border="1" cellpadding="5" cellspacing="0">
    <thead>
        <tr><th>Column Name</th><th>Description</th><th>Data Type</th></tr>
    </thead>
    <tbody>
        <tr><td>invoice_number</td><td>Foreign key referencing <code>dim_invoice</code></td><td>VARCHAR</td></tr>
        <tr><td>client_id</td><td>Foreign key referencing <code>dim_client</code></td><td>VARCHAR</td></tr>
        <tr><td>date_id</td><td>Foreign key referencing <code>dim_date</code></td><td>INT</td></tr>
        <tr><td>gross_sales</td><td>Total sales before discount</td><td>FLOAT</td></tr>
        <tr><td>discount</td><td>Discount amount</td><td>FLOAT</td></tr>
        <tr><td>net_sales</td><td>Sales after discount</td><td>FLOAT</td></tr>
        <tr><td>cog</td><td>Cost of goods sold</td><td>FLOAT</td></tr>
        <tr><td>tva</td><td>Value-added tax</td><td>FLOAT</td></tr>
        <tr><td>stamp_duty</td><td>Stamp duty tax</td><td>FLOAT</td></tr>
        <tr><td>total_amount</td><td>Final amount (net sales + taxes)</td><td>FLOAT</td></tr>
        <tr><td>profit</td><td>net_sales - cog</td><td>FLOAT</td></tr>
        <tr><td>client_category</td><td>Category name (e.g., Private Organization, Walk-in Client)</td><td>VARCHAR</td></tr>
    </tbody>
</table>

<h4>Dimension Table: <code>dim_client</code>Rows: ~2,804 (unique clients)</h4>
<table border="1" cellpadding="5" cellspacing="0">
    <thead>
        <tr><th>Column Name</th><th>Description</th><th>Data Type</th></tr>
    </thead>
    <tbody>
        <tr><td>client_id</td><td>Primary key for client</td><td>VARCHAR</td></tr>
    </tbody>
</table>

<h4>Dimension Table: <code>dim_date</code>Rows: ~1,058 (daily granularity for ~2.5 years, e.g., 2023–2025)</h4>
<table border="1" cellpadding="5" cellspacing="0">
    <thead>
        <tr><th>Column Name</th><th>Description</th><th>Data Type</th></tr>
    </thead>
    <tbody>
        <tr><td>date_id</td><td>Primary key for date (can be in YYYYMMDD format)</td><td>INT</td></tr>
        <tr><td>date</td><td>Full date</td><td>DATE</td></tr>
    </tbody>
</table>

<h4>Dimension Table: <code>dim_invoice</code> Rows: ~26,507 (one per invoice)</h4>
<table border="1" cellpadding="5" cellspacing="0">
    <thead>
        <tr><th>Column Name</th><th>Description</th><th>Data Type</th></tr>
    </thead>
    <tbody>
        <tr><td>invoice_number</td><td>Primary key for invoice</td><td>VARCHAR</td></tr>
        <tr><td>is_return</td><td>Flag to indicate if invoice is a return</td><td>VARCHAR</td></tr>
        <tr><td>payment_method</td><td>Payment method used</td><td>VARCHAR</td></tr>
    </tbody>
</table>
<hr>
